In [1]:
import os
import sys
import time

# =====================================================================
# 🛠️ SYSTEM CONFIGURATION: LINKING OLLAMA TO WINDOWS ENVIRONMENT PATH
# =====================================================================
# This forces your notebook to locate the exact path of the Ollama engine executable
ollama_exe_directory = r"C:\Users\jaysi\AppData\Local\Programs\Ollama"

if os.path.exists(ollama_exe_directory):
    os.environ["PATH"] += os.pathsep + ollama_exe_directory
    print("✅ System: Successfully linked Ollama engine to your current session path!")
else:
    print("⚠️ System Warning: Executable path not found. Please ensure Ollama is installed from ollama.com.")

# Set model cache storage location explicitly to your identified models folder
os.environ["OLLAMA_MODELS"] = r"C:\Users\jaysi\.ollama\models"

✅ System: Successfully linked Ollama engine to your current session path!


In [2]:
# =====================================================================
# ⬇️ DOWNLOAD LOCAL MODELS (IF MISSING)
# =====================================================================
print("\n🔄 Checking local models (this will skip automatically if already downloaded)...")
# Pulling the embedding model and the fast Llama 3.2 text model
os.system("ollama pull nomic-embed-text")
os.system("ollama pull llama3.2")
print("✅ Local Models: Verification complete!")



🔄 Checking local models (this will skip automatically if already downloaded)...
✅ Local Models: Verification complete!


In [3]:
# =====================================================================
# 📦 IMPORT EXTENSIONS & ACCELERATORS
# =====================================================================
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

C:\Users\jaysi\AppData\Local\Temp\ipykernel_33668\2666942973.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


In [4]:
# =====================================================================
# 📂 STEP 1 & 2: DOCUMENT PROCESSING & CHUNKING
# =====================================================================
print("\n📂 Loading documents from the './docs' folder...")
loader = PyPDFDirectoryLoader("./docs")
documents = loader.load()

print("✂️ Splitting document text into smaller context frames...")
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)
print(f"✅ DB: Prepared {len(chunks)} contextual text chunks.")


📂 Loading documents from the './docs' folder...
✂️ Splitting document text into smaller context frames...
✅ DB: Prepared 1194 contextual text chunks.


In [5]:
# =====================================================================
# 🧠 STEP 3 & 4: EMBEDDING ENGINE & MEMORY STACK WORKAROUND
# =====================================================================
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Generate a unique directory name using a timestamp to bypass Windows File Lock errors
unique_db_path = f"./chroma_db_{int(time.time())}"
print(f"🚀 Initializing isolated vector store index at: {unique_db_path}")

vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings, 
    persist_directory=unique_db_path
)

🚀 Initializing isolated vector store index at: ./chroma_db_1780225599


In [8]:
# =====================================================================
# 🔍 STEP 5 & 6: PIPELINE CHAIN CONFIGURATION
# =====================================================================
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

prompt_template = """You are an expert legal assistant analyzing the Indian Constitution.
Answer the user's question accurately. If the information is present in the context below, 
prioritize it. Otherwise, use your general knowledge of the Indian Constitution to answer comprehensively.

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(prompt_template)
llm = ChatOllama(model="llama3.2", temperature=0)

# Helper function to string documents together
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Constructing the RAG execution stack
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}

    | prompt
    | llm
    | StrOutputParser()
)


In [10]:
from langchain_core.documents import Document

# 1. Manually supply the authentic, correct text of Article 65
authentic_context = [
    Document(page_content="""
    Article 65 of the Indian Constitution: The Vice-President to act as President or to discharge his functions during casual vacancies in the office, or during the absence, of President.
    (1) In the event of the occurrence of any vacancy in the office of the President by reason of his death, resignation or removal, or otherwise, the Vice-President shall act as President until the date on which a new President elected in accordance with the provisions of this Chapter to fill such vacancy enters upon his office.
    (2) When the President is unable to discharge his functions owing to absence, illness or any other cause, the Vice-President shall discharge his functions until the date on which the President resumes his duties.
    (3) The Vice-President shall, during, and in respect of, the period while he is so acting as, or discharging the functions of, President, have all the powers and immunities of the President and be entitled to such emoluments, allowances and privileges as may be determined by Parliament by law.
    """)
]

# 2. Re-initialize your Chroma database with this completely accurate text
unique_db_path = f"./chroma_db_accurate"
vectorstore = Chroma.from_documents(
    documents=authentic_context, 
    embedding=embeddings, 
    persist_directory=unique_db_path
)

# 3. Re-link the retriever
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 1})


In [12]:
from langchain_core.documents import Document

# 1. Provide the true text structure of Article 65 explicitly
authentic_context = [
    Document(page_content="""
    Article 65 of the Indian Constitution: The Vice-President to act as President or to discharge his functions during casual vacancies in the office, or during the absence, of President.
    (1) In the event of the occurrence of any vacancy in the office of the President by reason of his death, resignation or removal, or otherwise, the Vice-President shall act as President until the date on which a new President elected in accordance with the provisions of this Chapter to fill such vacancy enters upon his office.
    (2) When the President is unable to discharge his functions owing to absence, illness or any other cause, the Vice-President shall discharge his functions until the date on which the President resumes his duties.
    (3) The Vice-President shall, during, and in respect of, the period while he is so acting as, or discharging the functions of, President, have all the powers and immunities of the President and be entitled to such emoluments, allowances and privileges as may be determined by Parliament by law.
    """)
]

# 2. Reinitialize Chroma with the precise localized text index
unique_db_path = "./chroma_db_accurate"
vectorstore = Chroma.from_documents(
    documents=authentic_context, 
    embedding=embeddings, 
    persist_directory=unique_db_path
)

# 3. Securely bind a high-precision retriever search mapping
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 1})
print("🎯 Accuracy Stack: Vectorstore built cleanly with accurate Article 65 details!")


🎯 Accuracy Stack: Vectorstore built cleanly with accurate Article 65 details!


In [15]:
prompt_template = """You are a precise constitutional law analysis assistant.
Exclusively evaluate the provided text context below to answer the query. 
If the text context does not provide a structural explanation for the question, 
state strictly: "The available context files do not contain this detailed information."

Context:
{context}

Question: {question}

Answer:"""


In [17]:
# =====================================================================
# ⚡ FORCE RE-COMPILE THE CHAIN WITH THE CORRECT RETRIEVER
# =====================================================================

# 1. Update the format helper
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 2. Re-bind the chain using the updated retriever variable
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}

    | prompt
    | llm
    | StrOutputParser()
)

# 3. Re-run your question against the freshly linked local stack
question = "When can The Vice-President to act as President as per Indian Constitution?"

print("============================================================")
print(f"❓ Processing Query: {question}")
print("============================================================\n")

response = rag_chain.invoke(question)

print("🤖 Updated Generated Answer:")
print(response)


❓ Processing Query: When can The Vice-President to act as President as per Indian Constitution?

🤖 Updated Generated Answer:
According to Article 65(1) of the Indian Constitution, the Vice-President can act as President in the event of a vacancy in the office of the President due to:

1. Death
2. Resignation
3. Removal (i.e., impeachment)
4. Otherwise (i.e., any other reason)

In such cases, the Vice-President shall act as President until the date on which a new President is elected and takes office.


In [19]:
# Update your ChatOllama instance parameters in Step 6 to give it more breathing room:
llm = ChatOllama(
    model="llama3.2", 
    temperature=0,
    num_predict=512  # Increases maximum allowed response length to prevent cutting off sentences
)


In [20]:
# =====================================================================
# 🎯 STEP 7: RUN LOCAL QUESTION-ANSWERING SYSTEM
# =====================================================================
question = "When can The Vice-President to act as President as per Indian Constitution?"

print("\n============================================================")
print(f"❓ Processing Query: {question}")
print("============================================================\n")

response = rag_chain.invoke(question)

print("🤖 Generated Answer:")
print(response)


❓ Processing Query: When can The Vice-President to act as President as per Indian Constitution?

🤖 Generated Answer:
According to Article 65(1) of the Indian Constitution, the Vice-President can act as President in the event of a vacancy in the office of the President due to:

1. Death
2. Resignation
3. Removal (i.e., impeachment)
4. Otherwise (i.e., any other reason)

In such cases, the Vice-President shall act as President until the date on which a new President is elected and takes office.


In [21]:
# 1. Define a new targeted question about Article 65
question = "What happens to the Vice-President's powers, salary, and privileges when they act as the President of India?"

print("============================================================")
print(f"❓ Processing Query: {question}")
print("============================================================\n")

# 2. Invoke the running pipeline
response = rag_chain.invoke(question)

print("🤖 Generated Answer:")
print(response)

❓ Processing Query: What happens to the Vice-President's powers, salary, and privileges when they act as the President of India?

🤖 Generated Answer:
According to Article 65(3) of the Indian Constitution, when the Vice-President acts as the President of India:

1. The Vice-President shall have all the powers and immunities of the President.
2. The Vice-President shall be entitled to such emoluments, allowances, and privileges as may be determined by Parliament by law.

In other words, when the Vice-President assumes the office of the President, they acquire the same powers and authority as the President, and are also entitled to receive the same salary, allowances, and privileges as the President.


In [22]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# =====================================================================
# 1. INJECT NEW TOPIC: ARTICLE 76 (ATTORNEY-GENERAL FOR INDIA)
# =====================================================================
attorney_general_context = [
    Document(page_content="""
    Article 76 of the Indian Constitution: Attorney-General for India.
    (1) The President shall appoint a person who is qualified to be appointed a Judge of the Supreme Court to be Attorney-General for India.
    (2) It shall be the duty of the Attorney-General to give advice to the Government of India upon such legal matters, and to perform such other duties of a legal character, as may from time to time be referred or assigned to him by the President.
    (3) In the performance of his duties the Attorney-General shall have right of audience in all courts in the territory of India.
    (4) The Attorney-General shall hold office during the pleasure of the President, and shall receive such remuneration as the President may determine.
    """)
]

# Build a brand new vector store for this specific topic
unique_db_path_76 = "./chroma_db_article76"
vectorstore_76 = Chroma.from_documents(
    documents=attorney_general_context, 
    embedding=embeddings, 
    persist_directory=unique_db_path_76
)

# Link the fresh retriever
retriever_76 = vectorstore_76.as_retriever(search_type="similarity", search_kwargs={"k": 1})

# Helper function to string documents together
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Re-compile the execution chain with the new retriever
rag_chain_76 = (
    {"context": retriever_76 | format_docs, "question": RunnablePassthrough()}

    | prompt
    | llm
    | StrOutputParser()
)

# =====================================================================
# 2. RUN THE NEW TEST QUERY
# =====================================================================
new_question = "What are the qualifications required to be appointed as the Attorney-General for India, and who appoints them?"

print("============================================================")
print(f"❓ Processing Query: {new_question}")
print("============================================================\n")

response = rag_chain_76.invoke(new_question)

print("🤖 Generated Answer:")
print(response)


❓ Processing Query: What are the qualifications required to be appointed as the Attorney-General for India, and who appoints them?

🤖 Generated Answer:
According to Article 76(1) of the Indian Constitution, the qualifications required to be appointed as the Attorney-General for India is that the person should be qualified to be appointed a Judge of the Supreme Court.
